### 🛠️ Initialize Notebook Variables

**Only modify entries under _USER CONFIGURATION_.**

In [ ]:
import utils
from typing import List

from apimtypes import API, APIM_SKU, GET_APIOperation, INFRASTRUCTURE, Region
from console import print_error, print_ok
from azure_resources import get_infra_rg_name

# ------------------------------
#    USER CONFIGURATION
# ------------------------------

rg_location   = Region.EAST_US_2
index         = 1
apim_sku      = APIM_SKU.DEVELOPER             # Cheapest VNet-capable SKU for appgw-apim; for appgw-apim-pe use STANDARDV2 or PREMIUMV2
deployment    = INFRASTRUCTURE.APPGW_APIM    # Options: see supported_infras below
api_prefix    = 'egress-'                     # ENTER A PREFIX FOR THE APIS TO REDUCE COLLISION POTENTIAL WITH OTHER SAMPLES
tags          = ['egress-control', 'networking'] # ENTER DESCRIPTIVE TAGS
apim_nsg_name = 'nsg-apim'                    # NSG attached to the APIM subnet; change to 'nsg-apim-strict' if deployed with strict NSGs



# ------------------------------
#    SYSTEM CONFIGURATION
# ------------------------------

sample_folder    = 'egress-control'
rg_name          = get_infra_rg_name(deployment, index)
supported_infras = [
    INFRASTRUCTURE.APPGW_APIM,
    INFRASTRUCTURE.APPGW_APIM_PE,
]
nb_helper        = utils.NotebookHelper(
    sample_folder,
    rg_name,
    rg_location,
    deployment,
    supported_infras,
    index = index,
    apim_sku = apim_sku
)

# Whether APIM is in VNet integration mode (V2 SKUs) vs VNet injection mode (V1 SKUs)
apim_vnet_integration = apim_sku.is_v2()

# Define the APIs and their operations and policies

# API 1: Weather Forecast — allowed via Azure Firewall (HTTPS to api.weather.gov)
pol_weather  = utils.read_policy_xml('weather-forecast.xml', sample_name = sample_folder)
weather_get  = GET_APIOperation(
    'Returns the Seattle weather forecast from api.weather.gov. HTTPS traffic is permitted through Azure Firewall.',
    pol_weather
)
weather_path = f'{api_prefix}weather'
weather_api  = API(
    weather_path,
    'Weather Forecast (Allowed)',
    weather_path,
    'Proxies requests to api.weather.gov over HTTPS. Permitted by the Azure Firewall application rule.',
    operations = [weather_get],
    tags       = tags,
    serviceUrl = 'https://api.weather.gov'
)

# API 2: Weather via HTTP — blocked by Azure Firewall (HTTP/port 80 not permitted)
blocked_http_get  = GET_APIOperation(
    'Attempts to reach api.weather.gov over HTTP (port 80). Azure Firewall denies port 80 outbound.'
)
blocked_http_path = f'{api_prefix}blocked-http'
blocked_http_api  = API(
    blocked_http_path,
    'Weather Forecast HTTP (Blocked)',
    blocked_http_path,
    'Proxies requests to api.weather.gov over HTTP. Blocked by Azure Firewall (no rule for port 80).',
    operations = [blocked_http_get],
    tags       = tags,
    serviceUrl = 'http://api.weather.gov'
)

# API 3: AccuWeather — blocked by Azure Firewall (FQDN not in the allow list)
blocked_host_get  = GET_APIOperation(
    'Attempts to reach api.accuweather.com over HTTPS. Azure Firewall denies this FQDN as it is not in the allow list.'
)
blocked_host_path = f'{api_prefix}blocked-host'
blocked_host_api  = API(
    blocked_host_path,
    'AccuWeather (Blocked)',
    blocked_host_path,
    'Proxies requests to api.accuweather.com over HTTPS. Blocked by Azure Firewall (FQDN not in allow list).',
    operations = [blocked_host_get],
    tags       = tags,
    serviceUrl = 'https://api.accuweather.com'
)

# APIs Array
apis: List[API] = [weather_api, blocked_http_api, blocked_host_api]

print_ok('Notebook initialized')

### 🚀 Deploy Infrastructure and APIs

Creates the Bicep deployment into the previously-specified resource group. A Bicep parameters, `params.json`, file will be created prior to execution.

This cell deploys:
- Hub VNet (`10.1.0.0/16`) with Azure Firewall Standard
- VNet peerings between hub and spoke
- Route table on the APIM subnet (`snet-apim`) to route internet traffic through the firewall
- Azure Firewall policy rules allowing `api.weather.gov` HTTPS and required APIM management traffic
- Three APIM APIs to test allowed and blocked internet access

> ⏱️ Azure Firewall deployment typically takes **8–12 minutes**.

In [ ]:
# Build the bicep parameters
bicep_parameters = {
    'apis':               {'value': [api.to_dict() for api in apis]},
    'apimVnetIntegration': {'value': apim_vnet_integration},
    'apimNsgName':        {'value': apim_nsg_name},
}

# Deploy the sample
output = nb_helper.deploy_sample(bicep_parameters)

if output.success:
    apim_name           = output.get('apimServiceName',        'APIM Service Name')
    apim_gateway_url    = output.get('apimResourceGatewayURL', 'APIM API Gateway URL')
    apim_apis           = output.getJson('apiOutputs',         'APIs')
    firewall_private_ip = output.get('firewallPrivateIpAddress', 'Firewall Private IP')
    firewall_public_ip  = output.get('firewallPublicIpAddress',  'Firewall Public IP')

    print_ok('Deployment completed successfully')
else:
    print_error('Deployment failed!')
    raise SystemExit(1)

### ✅ Verify Egress Control Behaviour

The three API calls below confirm that Azure Firewall is correctly enforcing outbound routing:

| Test | API | Expected | Reason |
|------|-----|----------|--------|
| 1 | `nva-weather` | ✅ 200 | HTTPS to `api.weather.gov` — allowed by firewall application rule |
| 2 | `nva-blocked-http` | ❌ 5xx | HTTP to `api.weather.gov` — no rule permits port 80 outbound |
| 3 | `nva-blocked-host` | ❌ 5xx | HTTPS to `api.accuweather.com` — FQDN not in the allow list |

In [ ]:
from apimrequests import ApimRequests
from apimtesting import ApimTesting

if 'apim_apis' not in locals():
    raise SystemExit(1)

# Initialize testing framework
tests = ApimTesting('Egress Control Sample Tests', sample_folder, nb_helper.deployment)

# Determine endpoint URL, headers, and TLS verification based on infrastructure type
endpoint_url, request_headers, allow_insecure_tls = utils.get_endpoint(deployment, rg_name, apim_gateway_url)

# ********** TEST EXECUTIONS **********

# Test 1: Weather forecast (HTTPS to api.weather.gov — allowed by firewall)
reqs   = ApimRequests(endpoint_url, apim_apis[0]['subscriptionPrimaryKey'], request_headers, allowInsecureTls = allow_insecure_tls)
result = reqs.singleGet(f'/{weather_path}', msg = 'Test 1: Weather API via HTTPS — should succeed (200).')
tests.verify(result is not None and 'periods' in result, True)

# Test 2: Blocked HTTP (port 80 to api.weather.gov — denied by firewall)
reqs.subscriptionKey = apim_apis[1]['subscriptionPrimaryKey']
result = reqs.singleGet(f'/{blocked_http_path}', msg = 'Test 2: Weather API via HTTP — should fail (5xx, port 80 blocked).')
tests.verify(result is None or 'periods' not in (result or ''), True)

# Test 3: Blocked host (HTTPS to api.accuweather.com — FQDN not in allow list)
reqs.subscriptionKey = apim_apis[2]['subscriptionPrimaryKey']
result = reqs.singleGet(f'/{blocked_host_path}', msg = 'Test 3: AccuWeather HTTPS — should fail (5xx, FQDN not in allow list).')
tests.verify(result is None or 'periods' not in (result or ''), True)

tests.print_summary()

print_ok('All done!')